# Fase 1: Clasificación CT — MosMedData

Este notebook replica el flujo de CXR sobre slices 2D de MosMedData, usando split por `study_id` para evitar fuga de datos entre cortes del mismo estudio.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))
os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / '.matplotlib_cache'))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.config import config
from src.data.ct_preprocessing import get_ct_dataframes
from src.data.datasets import CTDataset
from src.data.transforms import get_ct_transforms
from src.training.classification_experiment import (
    get_device,
    make_run_config,
    save_classification_artifacts,
    seed_everything,
    train_and_evaluate,
)

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 40)

In [ ]:
seed_everything(config.RANDOM_SEED)
device = get_device()
device

## 2. Parámetros

In [ ]:
RUN_MODE = 'full'
RESUME_EXISTING = True
ARCHITECTURES = ['resnet50', 'densenet121', 'efficientnet_b0']
BALANCE_STRATEGIES = ['baseline', 'weighted_ce', 'focal_loss']

LABEL_MAP = {'CT-0': 0, 'CT-1': 1, 'CT-2': 2, 'CT-3+': 3}
IDX_TO_LABEL = {idx: label for label, idx in LABEL_MAP.items()}

models_dir = config.MODELS_DIR / 'ct'
results_dir = PROJECT_ROOT / 'results' / 'classification' / 'ct'
models_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

print(f'RUN_MODE={RUN_MODE}')
print(f'RESUME_EXISTING={RESUME_EXISTING}')
print(f'device={device}')
print(f'architectures={ARCHITECTURES}')
print(f'balance_strategies={BALANCE_STRATEGIES}')


## 3. Metadata y split por estudio

In [ ]:
metadata_path = config.CT_DIR / 'processed_2d_slices' / 'labels_metadata.csv'
if not metadata_path.exists():
    raise FileNotFoundError(f'No existe {metadata_path}. Ejecuta primero src/data/ct_preprocessing.py')

ct_df = pd.read_csv(metadata_path)
train_df, val_df, test_df = get_ct_dataframes(ct_df, config.RANDOM_SEED)

split_summary = pd.concat([
    train_df.assign(split='train'),
    val_df.assign(split='val'),
    test_df.assign(split='test'),
]).groupby(['split', 'label']).size().unstack(fill_value=0)

display(split_summary)
display(split_summary.div(split_summary.sum(axis=1), axis=0).round(3))

study_overlap = {
    'train_val': len(set(train_df.study_id) & set(val_df.study_id)),
    'train_test': len(set(train_df.study_id) & set(test_df.study_id)),
    'val_test': len(set(val_df.study_id) & set(test_df.study_id)),
}
study_overlap

## 4. Entrenamiento de referencia

In [ ]:
sample_run_config = make_run_config('ct', ARCHITECTURES[0], BALANCE_STRATEGIES[0], RUN_MODE)
sample_artifact_name = f"{sample_run_config.experiment_name}_{sample_run_config.run_mode}"
sample_model_path = models_dir / f"{sample_artifact_name}.pt"
sample_history_path = results_dir / f"{sample_artifact_name}_history.csv"
sample_summary_path = results_dir / f"{sample_artifact_name}_summary.json"

if RESUME_EXISTING and sample_model_path.exists() and sample_summary_path.exists():
    print(f"Artefacto existente detectado: {sample_artifact_name}. Se reutiliza sin reentrenar.")
    sample_result = None
    sample_summary = json.loads(sample_summary_path.read_text())
    display(pd.DataFrame([sample_summary]))
else:
    sample_result = train_and_evaluate(
        run_config=sample_run_config,
        dataset_cls=CTDataset,
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        train_transform=get_ct_transforms(config.CT_IMAGE_SIZE, is_train=True),
        eval_transform=get_ct_transforms(config.CT_IMAGE_SIZE, is_train=False),
        label_map=LABEL_MAP,
        in_channels=1,
        device=device,
    )

    display(sample_result['split_summary'])
    display(pd.DataFrame(sample_result['history']))

## 5. Guardado de artefactos

In [ ]:
if sample_result is None:
    saved_paths = {
        'model': sample_model_path,
        'history': sample_history_path,
        'summary': sample_summary_path,
        'classification_report': results_dir / f"{sample_artifact_name}_classification_report.csv",
        'confusion_matrix': results_dir / f"{sample_artifact_name}_confusion_matrix.csv",
        'predictions': results_dir / f"{sample_artifact_name}_predictions.csv",
    }
else:
    saved_paths = save_classification_artifacts(
        sample_run_config,
        sample_result,
        LABEL_MAP,
        models_dir,
        results_dir,
    )
saved_paths

## 6. Matriz completa

In [ ]:
all_summaries = []

for architecture in ARCHITECTURES:
    for strategy in BALANCE_STRATEGIES:
        run_config = make_run_config('ct', architecture, strategy, RUN_MODE)
        artifact_name = f"{run_config.experiment_name}_{run_config.run_mode}"
        model_path = models_dir / f"{artifact_name}.pt"
        summary_path = results_dir / f"{artifact_name}_summary.json"
        print(f"
=== {artifact_name} ===")

        if RESUME_EXISTING and model_path.exists() and summary_path.exists():
            print('Saltado: artefactos full existentes detectados.')
            summary = json.loads(summary_path.read_text())
            all_summaries.append({
                'experiment': summary['experiment'],
                'architecture': summary['architecture'],
                'balance_strategy': summary['balance_strategy'],
                'accuracy': summary['accuracy'],
                'f1_macro': summary['f1_macro'],
                'f1_weighted': summary['f1_weighted'],
                'auc_roc_macro': summary.get('auc_roc_macro'),
            })
            continue

        result = train_and_evaluate(
            run_config=run_config,
            dataset_cls=CTDataset,
            train_df=train_df,
            val_df=val_df,
            test_df=test_df,
            train_transform=get_ct_transforms(config.CT_IMAGE_SIZE, is_train=True),
            eval_transform=get_ct_transforms(config.CT_IMAGE_SIZE, is_train=False),
            label_map=LABEL_MAP,
            in_channels=1,
            device=device,
        )
        save_classification_artifacts(run_config, result, LABEL_MAP, models_dir, results_dir)
        all_summaries.append({
            'experiment': run_config.experiment_name,
            'architecture': architecture,
            'balance_strategy': strategy,
            'accuracy': result['metrics']['accuracy'],
            'f1_macro': result['metrics']['f1_macro'],
            'f1_weighted': result['metrics']['f1_weighted'],
            'auc_roc_macro': result['metrics'].get('auc_roc_macro'),
        })

pd.DataFrame(all_summaries)


## 7. Curvas y matriz del experimento de referencia

In [ ]:
if sample_result is None:
    history_df = pd.read_csv(sample_history_path)
else:
    history_df = pd.DataFrame(sample_result['history'])
display(history_df)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
history_df[['train_loss', 'val_loss']].plot(ax=axes[0], marker='o')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
history_df[['train_acc', 'val_acc']].plot(ax=axes[1], marker='o')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
plt.tight_layout()

In [ ]:
if sample_result is None:
    confusion = pd.read_csv(saved_paths['confusion_matrix']).to_numpy()
else:
    confusion = sample_result['metrics']['confusion_matrix']
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    confusion,
    annot=True,
    fmt='d',
    cmap='Greens',
    xticklabels=[IDX_TO_LABEL[i] for i in range(len(IDX_TO_LABEL))],
    yticklabels=[IDX_TO_LABEL[i] for i in range(len(IDX_TO_LABEL))],
    ax=ax,
)
ax.set_xlabel('Predicción')
ax.set_ylabel('Real')
ax.set_title('Matriz de confusión CT')
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()